In [1]:
import json
from pathlib import Path
from datetime import datetime

PROC = Path("data/processed")
PROC.mkdir(parents=True, exist_ok=True)

preregistration = {
    "written_at": datetime.now().isoformat(),
    "written_before": "any fuzzy feature engineering",
    "fit_window":  {"start": "2022-03-12", "end": "2024-03-31"},
    "holdout_window": {"start": "2024-04-01", "end": "2024-07-02"},
    "evaluation_metric": "Harrell concordance index per task, averaged across 16 tasks",
    "model_change_rule": "baseline features vs baseline + 4 fuzzy IT2FIS membership features",
    "pre_registered_decision_rule": (
        "ΔC-index > 0 with paired block-bootstrap CI lower bound > 0: positive. "
        "CI contains 0: null. CI upper bound < 0: negative. "
        "No model variants after this registration."
    ),
    "winner_reference_afinit": 0.9142,
    "winner_per_task_scores": {
        "Borrow_Liquidated": 0.81, "Borrow_Deposit": 0.95, "Borrow_Repay": 0.91, "Borrow_Withdraw": 0.92,
        "Deposit_Liquidated": 0.82, "Deposit_Borrow": 0.97, "Deposit_Repay": 0.91, "Deposit_Withdraw": 0.95,
        "Repay_Liquidated": 0.81, "Repay_Borrow": 0.97, "Repay_Deposit": 0.94, "Repay_Withdraw": 0.91,
        "Withdraw_Liquidated": 0.89, "Withdraw_Borrow": 0.98, "Withdraw_Deposit": 0.98, "Withdraw_Repay": 0.93,
    },
}
with open(PROC / "pre_registration.json", "w") as f:
    json.dump(preregistration, f, indent=2)
print(f"pre-registration locked at {preregistration['written_at']}")
print(f"fit window:    {preregistration['fit_window']}")
print(f"holdout window: {preregistration['holdout_window']}")

pre-registration locked at 2026-10-15T00:34:28.886152
fit window:    {'start': '2022-03-12', 'end': '2024-03-31'}
holdout window: {'start': '2024-04-01', 'end': '2024-07-02'}


In [2]:
import time, gc
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from lifelines.utils import concordance_index
import warnings
warnings.filterwarnings("ignore")

RAW_ROOT  = Path("data/raw")
PROC_DIR  = Path("data/processed")
MODEL_DIR = Path("data/models/baseline_pinned")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with open(PROC_DIR / "task_registry.json") as f:
    TASKS = json.load(f)
with open(PROC_DIR / "feature_list.json") as f:
    FEAT = json.load(f)

FEATURES    = FEAT["features"]
CATEGORICAL = [c for c in ["userCoinTypeMode", "userReserveMode"] if c in FEATURES]

# Pre-registered split timestamps (seconds since epoch)
HOLDOUT_START = pd.Timestamp("2024-04-01").timestamp()
HOLDOUT_END   = pd.Timestamp("2024-07-02").timestamp()

In [3]:
def load_fit_and_holdout(task):
    train_path = RAW_ROOT / task["train_path"]
    cols = FEATURES + CATEGORICAL + ["timeDiff", "status", "timestamp"]
    cols = list(dict.fromkeys(cols))
    df = pd.read_csv(train_path, usecols=cols)

    # Encode categoricals
    for c in CATEGORICAL:
        le = LabelEncoder().fit(df[c].astype(str))
        df[c] = le.transform(df[c].astype(str))
    df[FEATURES] = df[FEATURES].fillna(0)

    fit     = df[df["timestamp"] <  HOLDOUT_START].reset_index(drop=True)
    holdout = df[(df["timestamp"] >= HOLDOUT_START) & (df["timestamp"] <= HOLDOUT_END)].reset_index(drop=True)
    return fit, holdout

In [4]:
PARAMS = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "verbosity": 0,
}

def xgb_cox_label(y_time, y_status):
    y = y_time.astype(float).copy()
    y[y_status == 0] = -y[y_status == 0]
    y[y == 0] = 1e-6
    return y

def fit_and_eval(fit, holdout, features):
    # Internal early-stopping validation: last 10% of fit by timestamp
    fit = fit.sort_values("timestamp").reset_index(drop=True)
    n = len(fit)
    n_val = int(n * 0.1)
    tr = fit.iloc[:n - n_val]
    vl = fit.iloc[n - n_val:]

    y_tr = xgb_cox_label(tr["timeDiff"].values, tr["status"].values)
    y_vl = xgb_cox_label(vl["timeDiff"].values, vl["status"].values)

    dtr = xgb.DMatrix(tr[features].values, label=y_tr)
    dvl = xgb.DMatrix(vl[features].values, label=y_vl)
    dho = xgb.DMatrix(holdout[features].values)

    bst = xgb.train(PARAMS, dtr, num_boost_round=300,
                    evals=[(dvl, "val")], early_stopping_rounds=20, verbose_eval=False)

    risk_ho = bst.predict(dho)
    c_ho = concordance_index(holdout["timeDiff"].values, -risk_ho, holdout["status"].values)
    return bst, risk_ho, c_ho, bst.best_iteration

In [5]:
results = []
t_start = time.time()

for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    print(f"[{i:2d}/16] {tid}  loading", end=" ")
    t0 = time.time()
    fit, holdout = load_fit_and_holdout(task)
    print(f"(fit {len(fit):,}, holdout {len(holdout):,}, {time.time()-t0:.0f}s)")

    print(f"       event rate fit {fit['status'].mean()*100:.2f}% | holdout {holdout['status'].mean()*100:.2f}%")
    print(f"       fitting", end=" ")
    t0 = time.time()
    bst, risk_ho, c_ho, best_iter = fit_and_eval(fit, holdout, FEATURES)
    dt = time.time() - t0
    print(f"({best_iter} rounds, holdout C={c_ho:.4f}, {dt:.0f}s)")

    bst.save_model(str(MODEL_DIR / f"{tid}.ubj"))
    # Save risk predictions on hold-out for later paired bootstrap
    pd.DataFrame({
        "risk": risk_ho,
        "timeDiff": holdout["timeDiff"].values,
        "status":   holdout["status"].values,
        "timestamp": holdout["timestamp"].values,
    }).to_csv(MODEL_DIR / f"{tid}_holdout_preds.csv", index=False)

    results.append({
        "task_id": tid,
        "n_fit": len(fit),
        "n_holdout": len(holdout),
        "fit_event_rate": float(fit["status"].mean()),
        "holdout_event_rate": float(holdout["status"].mean()),
        "holdout_c_index": float(c_ho),
        "best_iter": int(best_iter),
        "fit_seconds": dt,
    })

    del fit, holdout, bst
    gc.collect()

print(f"\ntotal time: {(time.time() - t_start)/60:.1f} min")

[ 1/16] Borrow_Deposit  loading (fit 784,167, holdout 98,041, 9s)
       event rate fit 43.04% | holdout 41.09%
       fitting (36 rounds, holdout C=0.7115, 10s)
[ 2/16] Borrow_Repay  loading (fit 757,265, holdout 88,543, 8s)
       event rate fit 84.26% | holdout 87.98%
       fitting (188 rounds, holdout C=0.8254, 24s)
[ 3/16] Borrow_Withdraw  loading (fit 785,318, holdout 98,106, 9s)
       event rate fit 33.99% | holdout 39.85%
       fitting (37 rounds, holdout C=0.7115, 11s)
[ 4/16] Borrow_Liquidated  loading (fit 786,960, holdout 98,420, 9s)
       event rate fit 2.84% | holdout 1.72%
       fitting (18 rounds, holdout C=0.8043, 9s)
[ 5/16] Deposit_Borrow  loading (fit 1,725,041, holdout 440,300, 23s)
       event rate fit 21.57% | holdout 7.77%
       fitting (70 rounds, holdout C=0.9260, 33s)
[ 6/16] Deposit_Repay  loading (fit 1,722,401, holdout 440,439, 22s)
       event rate fit 21.16% | holdout 7.73%
       fitting (299 rounds, holdout C=0.9548, 70s)
[ 7/16] Deposit_Withdr

In [6]:
res_df = pd.DataFrame(results)
res_df["fit_event_rate"]     = (res_df["fit_event_rate"] * 100).round(2)
res_df["holdout_event_rate"] = (res_df["holdout_event_rate"] * 100).round(2)
res_df["holdout_c_index"]    = res_df["holdout_c_index"].round(4)
res_df["fit_seconds"]        = res_df["fit_seconds"].round(0).astype(int)

print(res_df.to_string(index=False))
mean_c = res_df["holdout_c_index"].mean()
print(f"\nmean pre-registered hold-out C-index: {mean_c:.4f}")
print(f"Afinit winner reference (hidden test): 0.9142")
print(f"gap from winner: {0.9142 - mean_c:+.4f}")

res_df.to_csv(MODEL_DIR / "pinned_baseline_summary.csv", index=False)

            task_id   n_fit  n_holdout  fit_event_rate  holdout_event_rate  holdout_c_index  best_iter  fit_seconds
     Borrow_Deposit  784167      98041           43.04               41.09           0.7115         36           10
       Borrow_Repay  757265      88543           84.26               87.98           0.8254        188           24
    Borrow_Withdraw  785318      98106           33.99               39.85           0.7115         37           11
  Borrow_Liquidated  786960      98420            2.84                1.72           0.8043         18            9
     Deposit_Borrow 1725041     440300           21.57                7.77           0.9260         70           33
      Deposit_Repay 1722401     440439           21.16                7.73           0.9548        299           70
   Deposit_Withdraw 1714498     253694           49.07               41.35           0.8984        299           80
 Deposit_Liquidated 1727846     440679            6.77                2.